# Stuttering Detection: K-Nearest Neighbors (KNN) Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Distance-Based Classification on WavLM Manifolds

---

## Step 1: Initialization

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import KNNModel
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

# Paths to our distributed dataset
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
SAMPLE_LIMIT = None
STRICT_LABELS = True
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True, strict=STRICT_LABELS)
    
    # Native randomized sampling of the dataset
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Preparation Engine

In [2]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True, strict=STRICT_LABELS)
manager = DataManager(None, None)

# Loading .npy features
X, y = manager.load_from_folders(fluent_dir, disfluent_dir, limit=SAMPLE_LIMIT, label_dict=label_dict)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Oversampling training data only
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Anti-Leakage Standard Selection (MANDATORY for KNN accuracy)
X_train_final = manager.preprocess(X_train_bal, method="standard", fit=True)
X_test_final = manager.preprocess(X_test, method="standard", fit=False)

[DataManager] Quality Filter: Removed 3938 low-quality samples.


## Step 4: Model Execution - K-Nearest Neighbors
We use **K=5** neighbors.

In [3]:
knn_model = KNNModel("KNN_K5", n_neighbors=5)
knn_model.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
knn_model.evaluate(X_test_final, y_test)

[Model: KNN_K5] Initialized.
[KNN_K5] Stores 4440 training vectors...

--- Evaluation on Unseen Test Set ---

--- Evaluation: KNN_K5 ---
Accuracy: 0.5992
Precision: 0.4355
Recall: 0.4075
F1: 0.4211

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      336             140            
True: Stutter(1)     157             108            


{'accuracy': 0.5991902834008097,
 'precision': 0.43548387096774194,
 'recall': 0.4075471698113208,
 'f1': 0.42105263157894735,
 'confusion_matrix': array([[336, 140],
        [157, 108]])}

# Part 2: Hyperparameter Optimization
---

In [ ]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.neighbors import KNeighborsClassifier

# Prepare Cross-Validation Split (Combining Train + Val)
X_train_val = np.vstack((X_train_final, manager.preprocess(X_val, method="standard", fit=False)))
y_train_val = np.hstack((y_train_bal, y_val))

# CV Setup
split_indices = np.hstack((
    -1 * np.ones(len(X_train_final)),
    np.zeros(len(X_val))
))
ps = PredefinedSplit(test_fold=split_indices)

print("--- Tuning KNN (Number of Neighbors) ---")
knn_params = {
    'n_neighbors': [1, 3, 5, 7, 11, 15, 21, 31],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'cosine']
}

knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=ps, scoring='accuracy', n_jobs=-1)
knn_grid.fit(X_train_val, y_train_val)

print(f"Best KNN Params: {knn_grid.best_params_}")
print(f"Best KNN Val Accuracy: {knn_grid.best_score_:.4f}")

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

best_knn = knn_grid.best_estimator_
y_pred = best_knn.predict(X_test_final)

print("\n--- Final Optimized KNN Evaluation ---")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(best_knn, X_test_final, y_test, cmap='Blues', display_labels=['Fluent', 'Stutter'])
plt.title(f"Optimized KNN (Acc: {accuracy_score(y_test, y_pred):.4f})")
plt.show()